In [1]:
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

In [10]:
import pandas as pd

from gensim import utils
from gensim.models.fasttext import FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

from stemmer import Stemmer, tokenize

In [16]:
class TripletCorpusLoader:
    def __init__(self, corpus_path: str, vocab_path: str):
        self.corpus_path = corpus_path
        self.stemmer = None

        if vocab_path:
            self.stemmer = Stemmer(vocab_path)

    def __iter__(self):
        df_corpus = pd.read_json(self.corpus_path, lines=True)
        corpus = df_corpus.values.ravel().tolist()
        
        for line in corpus:
            if self.stemmer:
                line = self.stem_sentence(line)

            yield utils.simple_preprocess(line)

    def stem_sentence(self, text: str) -> str:
        return " ".join([self.stemmer.stem_ams(t) for t in tokenize(text)])

corpus_loader = TripletCorpusLoader("../data/triplet/triplet.jsonl", "../data/sundabaru1-vocab.txt")

## Doc2Vec

In [17]:
train_corpus = [TaggedDocument(x, [i]) for i, x in enumerate(corpus_loader.__iter__())]

In [5]:
model = Doc2Vec(vector_size=50, min_count=2, epochs=40)
model.build_vocab(train_corpus)
model

2025-04-26 07:49:38,154 : INFO : Doc2Vec lifecycle event {'params': 'Doc2Vec<dm/m,d50,n5,w5,mc2,s0.001,t3>', 'datetime': '2025-04-26T07:49:38.154548', 'gensim': '4.3.2', 'python': '3.11.12 (main, Apr  9 2025, 04:04:00) [Clang 20.1.0 ]', 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'event': 'created'}
2025-04-26 07:49:38,155 : INFO : collecting all words and their counts
2025-04-26 07:49:38,155 : INFO : PROGRESS: at example #0, processed 0 words (0 words/s), 0 word types, 0 tags
2025-04-26 07:49:38,179 : INFO : PROGRESS: at example #10000, processed 122096 words (5362034 words/s), 11703 word types, 0 tags
2025-04-26 07:49:38,198 : INFO : PROGRESS: at example #20000, processed 235757 words (6058319 words/s), 17543 word types, 0 tags
2025-04-26 07:49:38,207 : INFO : collected 18554 word types and 22476 unique tags from a corpus of 22476 examples and 264009 words
2025-04-26 07:49:38,208 : INFO : Creating a fresh vocabulary
2025-04-26 07:49:38,231 : INFO : D

In [6]:
model.train(train_corpus, total_examples=model.corpus_count, epochs=model.epochs)

2025-04-26 07:49:40,903 : INFO : Doc2Vec lifecycle event {'msg': 'training model with 3 workers on 10898 vocabulary and 50 features, using sg=0 hs=0 sample=0.001 negative=5 window=5 shrink_windows=True', 'datetime': '2025-04-26T07:49:40.903668', 'gensim': '4.3.2', 'python': '3.11.12 (main, Apr  9 2025, 04:04:00) [Clang 20.1.0 ]', 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'event': 'train'}
2025-04-26 07:49:41,929 : INFO : EPOCH 0 - PROGRESS: at 39.81% examples, 97330 words/s, in_qsize 6, out_qsize 0
2025-04-26 07:49:42,966 : INFO : EPOCH 0 - PROGRESS: at 86.30% examples, 100982 words/s, in_qsize 4, out_qsize 0
2025-04-26 07:49:43,091 : INFO : EPOCH 0: training on 264009 raw words (238791 effective words) took 2.2s, 109386 effective words/s
2025-04-26 07:49:44,100 : INFO : EPOCH 1 - PROGRESS: at 47.38% examples, 117091 words/s, in_qsize 6, out_qsize 0
2025-04-26 07:49:45,121 : INFO : EPOCH 1 - PROGRESS: at 92.16% examples, 109016 words/s, in_qsize 2, o

In [7]:
model.save("../models/ams_doc2vec-stem.model")

2025-04-26 07:51:25,523 : INFO : Doc2Vec lifecycle event {'fname_or_handle': '../models/ams_doc2vec-stem.model', 'separately': 'None', 'sep_limit': 10485760, 'ignore': frozenset(), 'datetime': '2025-04-26T07:51:25.523838', 'gensim': '4.3.2', 'python': '3.11.12 (main, Apr  9 2025, 04:04:00) [Clang 20.1.0 ]', 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'event': 'saving'}
2025-04-26 07:51:25,525 : INFO : not storing attribute cum_table
2025-04-26 07:51:25,538 : INFO : saved ../models/ams_doc2vec-stem.model


In [8]:
model.infer_vector(['henteu', 'sampurna', 'pamulang', 'samemehna'])

array([ 0.21368562,  0.08152049,  0.10664392, -0.3956324 ,  0.03685253,
       -0.08280052, -0.09949838, -0.00285868, -0.01697302, -0.34828332,
       -0.26120707, -0.2788407 , -0.14778109, -0.18942659,  0.01588883,
        0.29586068,  0.53237146, -0.32177415,  0.2679571 ,  0.2792271 ,
       -0.02054232,  0.5411089 ,  0.25577152, -0.31034914, -0.01495226,
       -0.3252895 ,  0.30408671,  0.23398075, -0.37261218,  0.09783987,
        0.36006317,  0.0818508 ,  0.29866904,  0.21062304, -0.22720298,
       -0.20623727, -0.10333628, -0.09904645,  0.04742035,  0.01754012,
        0.1668752 ,  0.17553216, -0.16702859,  0.08270558,  0.19223642,
       -0.42814162,  0.25408998,  0.00252233, -0.07646338,  0.08107956],
      dtype=float32)

## FastText

In [18]:
train_corpus = list(corpus_loader.__iter__())
train_corpus[:5]

[['kumaha', 'cara', 'hontal', 'kahayang', 'dina', 'hirup'],
 ['hiji',
  'urang',
  'kedah',
  'sabar',
  'sareng',
  'henteu',
  'janten',
  'hambar',
  'dina',
  'usaha',
  'sabab',
  'balukar',
  'janglar',
  'pikeun',
  'hontal',
  'kahayang',
  'peryogi',
  'usaha',
  'anu',
  'tekun'],
 ['abdi',
  'dangu',
  'seueur',
  'warta',
  'ngeunan',
  'jalma',
  'anu',
  'suksés',
  'tapi',
  'henteu',
  'pernah',
  'nguping',
  'kumaha',
  'cara',
  'maranéhna',
  'ngaliwatan',
  'rintangan'],
 ['non', 'anu', 'kedah', 'laku', 'lamun', 'gering'],
 ['lamun',
  'gering',
  'ulah',
  'rungsing',
  'sabab',
  'pikir',
  'positif',
  'penting',
  'tong',
  'sieun',
  'tetep',
  'tenang',
  'da',
  'cageur',
  'tangtos',
  'bakal',
  'datang']]

In [19]:
model = FastText(vector_size=100)
model.build_vocab(corpus_iterable=train_corpus)
model.train(corpus_iterable=train_corpus, total_examples=len(train_corpus), epochs=10)
model

2025-04-26 08:00:55,492 : INFO : FastText lifecycle event {'params': 'FastText<vocab=0, vector_size=100, alpha=0.025>', 'datetime': '2025-04-26T08:00:55.492494', 'gensim': '4.3.2', 'python': '3.11.12 (main, Apr  9 2025, 04:04:00) [Clang 20.1.0 ]', 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'event': 'created'}
2025-04-26 08:00:55,493 : INFO : collecting all words and their counts
2025-04-26 08:00:55,494 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2025-04-26 08:00:55,531 : INFO : PROGRESS: at sentence #10000, processed 122096 words, keeping 11703 word types
2025-04-26 08:00:55,560 : INFO : PROGRESS: at sentence #20000, processed 235757 words, keeping 17543 word types
2025-04-26 08:00:55,567 : INFO : collected 18554 word types from a corpus of 264009 raw words and 22476 sentences
2025-04-26 08:00:55,568 : INFO : Creating a fresh vocabulary
2025-04-26 08:00:55,583 : INFO : FastText lifecycle event {'msg': 'effective_min_coun

In [21]:
model.wv.get_sentence_vector('henteu sampurna pamulang samemehna')

array([-1.92175957e-03, -1.45268142e-02,  5.44010727e-05,  1.91157088e-02,
       -1.77544262e-02, -1.37025453e-02,  2.35054009e-02,  1.28996130e-02,
       -1.17940810e-02,  3.75106484e-02,  2.25217547e-02, -3.96266058e-02,
       -3.79135075e-04, -8.16182047e-03, -2.06180243e-03,  3.41646150e-02,
        2.40429062e-02,  4.88161631e-02, -1.85051821e-02, -1.11801839e-02,
       -5.60490713e-02,  1.01287104e-02,  2.92863920e-02,  7.12546334e-02,
       -1.58659033e-02, -6.27809688e-02, -5.47111332e-02,  8.03954154e-03,
       -8.18065032e-02, -4.14637737e-02,  9.32510570e-03, -1.68387499e-02,
        8.82300660e-02,  4.21873778e-02,  4.37148884e-02,  2.25323141e-02,
        3.47060300e-02,  3.98135372e-02, -1.46676460e-02, -4.25694473e-02,
       -3.22192684e-02, -4.77346778e-02, -1.13392677e-02,  1.17284153e-02,
       -4.87445435e-03,  3.06118559e-02, -7.22082630e-02,  1.92451701e-02,
        1.87412603e-03,  8.54702573e-03,  7.02219433e-04,  5.64195551e-02,
       -3.13349739e-02,  